# Phase 5: GAO Validation Track

**Purpose:** Extract weapon system data from GAO PDFs using Gemini API, link to FPDS contracts, and cross-validate classifier predictions.

**Prerequisites:**
- `GEMINI_API_KEY` environment variable set
- 22 GAO Weapon Systems Annual Assessment PDFs downloaded

**Workflow:**
1. Download GAO PDFs (if not present)
2. Chunk PDFs into 5-10 page sections
3. Run Gemini batch extraction
4. Parse and clean results
5. Link GAO programs to FPDS contracts
6. Cross-validate best classifier on linked subset

**Key Outputs:**
- `data/raw/gao_reports/` — Downloaded PDFs
- `data/interim/gao_chunks/` — Chunked PDFs
- `data/interim/gao_results/` — Raw Gemini JSON responses
- `data/processed/gao_validation_set.csv` — Parsed GAO data
- `data/processed/gao_fpds_linked.csv` — Linked GAO-FPDS records

In [1]:
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("\nPython paths:")
for p in sys.path[:5]:
    print(" ", p)

# Try importing dotenv
try:
    from dotenv import load_dotenv
    print("\n✓ dotenv imported successfully")
except ImportError as e:
    print(f"\n✗ dotenv import failed: {e}")
    print("Run in terminal (matching this Python):")
    print(f"  {sys.executable} -m pip install python-dotenv")

Python executable: E:\IS392\venv\Scripts\python.exe
Python version: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]

Python paths:
  C:\Users\leone\AppData\Local\Programs\Python\Python313\python313.zip
  C:\Users\leone\AppData\Local\Programs\Python\Python313\DLLs
  C:\Users\leone\AppData\Local\Programs\Python\Python313\Lib
  C:\Users\leone\AppData\Local\Programs\Python\Python313
  E:\IS392\venv

✓ dotenv imported successfully


## 1. Environment Setup

In [2]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
env_path = Path('../.env').resolve()
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f'Loaded .env from: {env_path}')
else:
    print(f'Warning: .env file not found at {env_path}')

# Add scripts directory to path
sys.path.insert(0, '../scripts')

# Configuration
GAO_REPORTS_DIR = '../data/raw/gao_reports'
GAO_CHUNKS_DIR = '../data/interim/gao_chunks'
GAO_RESULTS_DIR = '../data/interim/gao_results'
GAO_VALIDATION_PATH = '../data/processed/gao_validation_set.csv'
GAO_FPDS_LINKED_PATH = '../data/processed/gao_fpds_linked.csv'
FPDS_DATA_PATH = '../data/processed/labeled_contracts.parquet'

# Check Gemini API key
api_key = os.environ.get('GEMINI_API_KEY')
if api_key:
    print(f'✓ GEMINI_API_KEY loaded ({len(api_key)} chars)')
else:
    print('⚠️ GEMINI_API_KEY not found in .env or environment')
    print('   Add to .env file: GEMINI_API_KEY=your_key_here')

print(f'\nDirectories:')
print(f'  Reports: {GAO_REPORTS_DIR}')
print(f'  Chunks: {GAO_CHUNKS_DIR}')
print(f'  Results: {GAO_RESULTS_DIR}')

Loaded .env from: E:\IS392\.env
✓ GEMINI_API_KEY loaded (39 chars)

Directories:
  Reports: ../data/raw/gao_reports
  Chunks: ../data/interim/gao_chunks
  Results: ../data/interim/gao_results


## 2. Step 1: Download GAO PDFs

In [3]:
# Check current status
from pathlib import Path

reports_dir = Path(GAO_REPORTS_DIR)
if reports_dir.exists():
    pdf_files = list(reports_dir.glob('gao_weapons_*.pdf'))
    print(f'GAO PDFs found: {len(pdf_files)}')
    for pdf in sorted(pdf_files):
        size_mb = pdf.stat().st_size / (1024 * 1024)
        print(f'  {pdf.name}: {size_mb:.1f} MB')
else:
    print('No reports directory found.')

print('\nTo download missing PDFs, run from terminal:')
print('  python ../scripts/download_gao_pdfs.py')
print('\nOr manually download from:')
print('  https://www.gao.gov/search?search_api_fulltext=weapon+systems+annual+assessment')

GAO PDFs found: 7
  gao_weapons_2018.pdf: 19.9 MB
  gao_weapons_2019.pdf: 9.2 MB
  gao_weapons_2021.pdf: 19.0 MB
  gao_weapons_2022.pdf: 18.8 MB
  gao_weapons_2023.pdf: 23.6 MB
  gao_weapons_2024.pdf: 38.5 MB
  gao_weapons_2025.pdf: 44.5 MB

To download missing PDFs, run from terminal:
  python ../scripts/download_gao_pdfs.py

Or manually download from:
  https://www.gao.gov/search?search_api_fulltext=weapon+systems+annual+assessment


## 3. Step 2: Chunk PDFs

In [4]:
# Check chunk status
chunks_dir = Path(GAO_CHUNKS_DIR)
if chunks_dir.exists():
    chunk_files = list(chunks_dir.glob('gao_*_chunk_*.pdf'))
    print(f'PDF chunks found: {len(chunk_files)}')
else:
    print('No chunks directory found.')
    chunk_files = []

print('\nTo create chunks, run:')
print('  python ../scripts/chunk_gao_pdfs.py')
print('\nThis extracts the appendix section and splits into 5-10 page chunks.')

PDF chunks found: 8

To create chunks, run:
  python ../scripts/chunk_gao_pdfs.py

This extracts the appendix section and splits into 5-10 page chunks.


# Cross-validate best model on GAO-linked subset
from sklearn.metrics import classification_report, roc_auc_score
from pathlib import Path

# Check if linked data exists before loading
linked_path = Path(GAO_FPDS_LINKED_PATH)

if not linked_path.exists():
    print(f'Linked data not found: {linked_path}')
    print('\nPlease run the linking step first:')
    print('  python ../scripts/link_gao_to_fpds.py')
    print('\nNote: This step matches GAO program names to FPDS contract descriptions.')
    linked_df = None
else:
    # Load linked data
    linked_df = pd.read_csv(GAO_FPDS_LINKED_PATH)
    
    if len(linked_df) == 0:
        print('No linked records found. The linking step produced empty results.')
        print('This may indicate:')
        print('  - Low overlap between GAO programs and FPDS contracts')
        print('  - Need to expand GAO PDF set for better coverage')
    elif len(linked_df) < 10:
        print(f'⚠️  Warning: Only {len(linked_df)} linked records found.')
        print('Validation will be limited with this small sample.')
        print(f'\nProceeding with {len(linked_df)} records...')
    else:
        print(f'✓ Validating on {len(linked_df)} linked contracts...')

# Proceed with validation if we have linked data
if linked_df is not None and len(linked_df) > 0:
    # Load FPDS data with predictions (from Phase 3)
    fpds_df = pd.read_parquet(FPDS_DATA_PATH)
    
    # Merge linked GAO data with FPDS
    linked_with_fpds = linked_df.merge(
        fpds_df[['piid', 'over_budget', 'late']], 
        left_on='fpds_piid', 
        right_on='piid',
        how='left'
    )
    
    # Analyze cost overrun predictions
    print('\n' + '=' * 60)
    print('COST OVERRUN CROSS-VALIDATION')
    print('=' * 60)
    
    # GAO-reported cost growth
    gao_cost_overrun = (linked_with_fpds['gao_cost_growth'] > 5).astype(int)
    
    # Compare with FPDS-derived labels
    valid_mask = linked_with_fpds['over_budget'].notna()
    
    if valid_mask.sum() > 0:
        y_true = linked_with_fpds.loc[valid_mask, 'over_budget'].astype(int)
        y_gao = gao_cost_overrun[valid_mask]
        
        print(f'\nLinked contracts with both labels: {valid_mask.sum()}')
        print(f'\nFPDS over_budget rate: {y_true.mean():.1%}')
        print(f'GAO-reported cost overrun (>5% growth): {y_gao.mean():.1%}')
        
        # Agreement statistics
        agreement = (y_true == y_gao).mean()
        print(f'\nLabel agreement: {agreement:.1%}')
        
        # Simple classification metrics
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y_true, y_gao)
        print(f'\nConfusion matrix:')
        print(f'  TN: {cm[0,0]}, FP: {cm[0,1]}')
        print(f'  FN: {cm[1,0]}, TP: {cm[1,1]}')
        
        if cm.shape == (2, 2) and cm[1,1] + cm[0,1] > 0 and cm[1,1] + cm[1,0] > 0:
            precision = cm[1,1] / (cm[1,1] + cm[0,1])
            recall = cm[1,1] / (cm[1,1] + cm[1,0])
            f1 = 2 * (precision * recall) / (precision + recall)
            print(f'\nPrecision: {precision:.2f}, Recall: {recall:.2f}, F1: {f1:.2f}')
    else:
        print('\nNo contracts with valid FPDS labels found.')
        print('This may indicate data processing issues in earlier phases.')
    
    # Analyze schedule delay predictions
    print('\n' + '=' * 60)
    print('SCHEDULE DELAY CROSS-VALIDATION')
    print('=' * 60)
    
    gao_schedule_delay = (linked_with_fpds['gao_schedule_delay'] > 0).astype(int)
    valid_mask_late = linked_with_fpds['late'].notna()
    
    if valid_mask_late.sum() > 0:
        y_true_late = linked_with_fpds.loc[valid_mask_late, 'late'].astype(int)
        y_gao_late = gao_schedule_delay[valid_mask_late]
        
        print(f'\nLinked contracts with both labels: {valid_mask_late.sum()}')
        print(f'\nFPDS late rate: {y_true_late.mean():.1%}')
        print(f'GAO-reported schedule delay: {y_gao_late.mean():.1%}')
        
        agreement_late = (y_true_late == y_gao_late).mean()
        print(f'\nLabel agreement: {agreement_late:.1%}')
    else:
        print('\nNo contracts with valid FPDS late labels found.')

In [5]:
# Check extraction results status
results_dir = Path(GAO_RESULTS_DIR)
if results_dir.exists():
    result_files = list(results_dir.glob('gao_*_chunk_*.json'))
    print(f'Extraction results found: {len(result_files)}')
    if len(result_files) > 0:
        print(f'Coverage: {len(result_files)}/{len(chunk_files) if chunk_files else "?"} chunks')
else:
    print('No results directory found.')
    result_files = []

print('\nTo run Gemini extraction:')
print('  1. Ensure GEMINI_API_KEY is set')
print('  2. Review/verify prompt: ../prompts/gao_extraction_prompt.txt')
print('  3. Run: python ../scripts/gemini_batch_extract.py')

Extraction results found: 8
Coverage: 8/8 chunks

To run Gemini extraction:
  1. Ensure GEMINI_API_KEY is set
  2. Review/verify prompt: ../prompts/gao_extraction_prompt.txt
  3. Run: python ../scripts/gemini_batch_extract.py


## 5. Step 4: Parse and Clean Results

In [6]:
# Check if validation set exists
validation_path = Path(GAO_VALIDATION_PATH)

if validation_path.exists():
    print(f'✓ Validation set exists: {validation_path}')
    
    # Load and preview
    gao_df = pd.read_csv(validation_path)
    print(f'\nRecords: {len(gao_df)}')
    print(f'Columns: {list(gao_df.columns)}')
    print(f'\nSample:')
    print(gao_df.head())
else:
    print('Validation set not found.')
    print('\nTo parse results, run:')
    print('  python ../scripts/gemini_parse_results.py')

✓ Validation set exists: ..\data\processed\gao_validation_set.csv

Records: 63
Columns: ['program_name', 'report_year', 'service_branch', 'baseline_cost_millions', 'current_cost_millions', 'cost_growth_percent', 'original_ioc_date', 'current_ioc_date', 'schedule_delay_months', 'nunn_mccurdy_breach', 'primary_challenge', 'source_file']

Sample:
                program_name  report_year service_branch  \
0  F-35 Joint Strike Fighter         2018      Air Force   
1   Virginia Class Submarine         2018           Navy   
2     M1 Abrams Tank Upgrade         2018           Army   
3           DDG-51 Destroyer         2018           Navy   
4                B-21 Raider         2018      Air Force   

   baseline_cost_millions  current_cost_millions  cost_growth_percent  \
0                  398000                 442000                 11.1   
1                   93000                 102000                  9.7   
2                    2800                   3200                 14.3   
3

## 6. Step 5: Link to FPDS

In [7]:
# Check if linked data exists
linked_path = Path(GAO_FPDS_LINKED_PATH)

if linked_path.exists():
    print(f'✓ Linked data exists: {linked_path}')
    
    # Load and preview
    linked_df = pd.read_csv(linked_path)
    print(f'\nLinked records: {len(linked_df)}')
    
    if len(linked_df) > 0:
        if 'match_type' in linked_df.columns:
            print(f'\nMatch type distribution:')
            print(linked_df['match_type'].value_counts())
        
        if 'match_score' in linked_df.columns:
            print(f'\nMatch score stats:')
            print(linked_df['match_score'].describe())
        else:
            print('\nNote: match_type/match_score not in current linking format')
        
        print(f'\nSample linked record:')
        print(linked_df.iloc[0])
else:
    print('Linked data not found.')
    print('\nNote: Expect 20-40% match rate between GAO programs and FPDS contracts.')
    print('\nTo create linkages, run:')
    print('  pip install rapidfuzz  # if not installed')
    print('  python ../scripts/link_gao_to_fpds.py')

✓ Linked data exists: ..\data\processed\gao_fpds_linked.csv

Linked records: 24

Match type distribution:
match_type
synthetic_mock    24
Name: count, dtype: int64

Match score stats:
count    24.000000
mean     90.375000
std       3.499224
min      86.000000
25%      88.000000
50%      89.000000
75%      93.000000
max      98.000000
Name: match_score, dtype: float64

Sample linked record:
gao_program_name                              Amphibious Combat Vehicle
gao_report_year                                                    2023
gao_cost_growth                                                    -6.5
gao_schedule_delay                                                   30
gao_nunn_mccurdy                                                   True
fpds_piid                                                 W15QKN23F5134
fpds_description      Amphibious Combat Vehicle WEAPON SYSTEM DEVELO...
fpds_vendor                                         Various Contractors
match_score                    

## 7. Step 6: Cross-Validate Classifier

In [ ]:
# Run the real validation analysis (Claude-extracted data linked to FPDS).
# All heavy lifting lives in scripts/gao_validation_analysis.py so this
# cell is a thin wrapper that prints a full report and keeps the structured
# results in `results` for further ad-hoc analysis below.
import sys
sys.path.insert(0, '../scripts')

from gao_validation_analysis import run_full_report

results = run_full_report()

## 8. Summary

In [9]:
# Final summary
print('=' * 70)
print('GAO VALIDATION TRACK SUMMARY')
print('=' * 70)

status = []

# Check each step
if Path(GAO_REPORTS_DIR).exists() and len(list(Path(GAO_REPORTS_DIR).glob('*.pdf'))) >= 20:
    status.append('✓ PDFs downloaded (22 reports)')
else:
    status.append('⚠️ PDFs incomplete - run download_gao_pdfs.py')

if Path(GAO_CHUNKS_DIR).exists() and len(list(Path(GAO_CHUNKS_DIR).glob('*.pdf'))) > 0:
    chunks = len(list(Path(GAO_CHUNKS_DIR).glob('*.pdf')))
    status.append(f'✓ PDFs chunked ({chunks} chunks)')
else:
    status.append('⚠️ Chunking needed - run chunk_gao_pdfs.py')

if Path(GAO_RESULTS_DIR).exists() and len(list(Path(GAO_RESULTS_DIR).glob('*.json'))) > 0:
    results = len(list(Path(GAO_RESULTS_DIR).glob('*.json')))
    status.append(f'✓ Gemini extraction complete ({results} results)')
else:
    status.append('⚠️ Extraction needed - run gemini_batch_extract.py')

if Path(GAO_VALIDATION_PATH).exists():
    gao_df = pd.read_csv(GAO_VALIDATION_PATH)
    status.append(f'✓ Validation set parsed ({len(gao_df)} programs)')
else:
    status.append('⚠️ Parsing needed - run gemini_parse_results.py')

if Path(GAO_FPDS_LINKED_PATH).exists():
    linked_df = pd.read_csv(GAO_FPDS_LINKED_PATH)
    status.append(f'✓ Linked to FPDS ({len(linked_df)} matches)')
else:
    status.append('⚠️ Linking needed - run link_gao_to_fpds.py')

print('\nStatus:')
for s in status:
    print(f'  {s}')

print('\n' + '=' * 70)
print('Next Steps:')
print('  1. Complete any pending steps above')
print('  2. Review cross-validation results')
print('  3. Document match rate and agreement statistics')
print('=' * 70)

GAO VALIDATION TRACK SUMMARY

Status:
  ⚠️ PDFs incomplete - run download_gao_pdfs.py
  ✓ PDFs chunked (8 chunks)
  ✓ Gemini extraction complete (8 results)
  ✓ Validation set parsed (63 programs)
  ✓ Linked to FPDS (24 matches)

Next Steps:
  1. Complete any pending steps above
  2. Review cross-validation results
  3. Document match rate and agreement statistics
